# v2 00 debug bbox distribution

draw raw bbox overlays in image coordinates before projection. each label is shown in a separate figure.


In [ ]:
from pathlib import Path
import ast
import re
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import v2_config


## input and plot settings


In [ ]:
INPUT_CSV = v2_config.V2_RAW_BBOX_CSV
MANUAL_ANNOTATION_CSV = PROJECT_ROOT.parent / 'ITP' / 'bbox_review' / 'manual_annotations.csv'
OUTPUT_DIR = v2_config.V2_ROOT_DIR / 'debug' / 'bbox_distribution_by_label'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_WIDTH = 640
IMAGE_HEIGHT = 480
RAW_BBOX_X_OFFSET = 70
MAX_BOXES_PER_LABEL = 3000  # set None to draw every bbox for each label.
RANDOM_STATE = 42

THERMAL_IMAGE_DIRS = [
    PROJECT_ROOT / 'thermal_images',
    PROJECT_ROOT.parent / 'raw_image',
]

HOTSPOT_DIRS = [
    v2_config.HOTSPOT_MACHINES_DIR,
    v2_config.HOTSPOT_PEOPLE_DIR,
]
HOTSPOT_LABEL_MAP = {'Cableduct': 'Light'}

print(f'input csv: {INPUT_CSV}')
print(f'raw bbox x offset: {RAW_BBOX_X_OFFSET}px')
print(f'manual annotation csv: {MANUAL_ANNOTATION_CSV}')
print(f'output dir: {OUTPUT_DIR}')


## load raw bbox data


In [ ]:
raw_df = pd.read_csv(INPUT_CSV)

bbox_df = raw_df.rename(
    columns={
        'filename': 'image_name',
        'class_name': 'label',
        'box_x0': 'bbox_x0',
    }
).copy()

bbox_columns = ['bbox_x0', 'bbox_y0', 'bbox_x1', 'bbox_y1']
base_columns = ['image_name', 'timestamp', 'label', *bbox_columns, 'temp_mean_c', 'temp_max_c', 'confidence']
bbox_df = bbox_df[base_columns].copy()

for column in bbox_columns + ['temp_mean_c', 'temp_max_c', 'confidence']:
    bbox_df[column] = pd.to_numeric(bbox_df[column], errors='coerce')

bbox_df = bbox_df.dropna(subset=['label', *bbox_columns])
# raw bbox source was cropped by 70 px on the x axis; add it back for 640x480 image-space overlay.
bbox_df['bbox_x0'] = bbox_df['bbox_x0'] + RAW_BBOX_X_OFFSET
bbox_df['bbox_x1'] = bbox_df['bbox_x1'] + RAW_BBOX_X_OFFSET
bbox_df['bbox_width'] = bbox_df['bbox_x1'] - bbox_df['bbox_x0']
bbox_df['bbox_height'] = bbox_df['bbox_y1'] - bbox_df['bbox_y0']
bbox_df['center_x'] = (bbox_df['bbox_x0'] + bbox_df['bbox_x1']) / 2
bbox_df['center_y'] = (bbox_df['bbox_y0'] + bbox_df['bbox_y1']) / 2

print(f'raw rows: {len(raw_df):,}')
print(f'usable bbox rows: {len(bbox_df):,}')
display(bbox_df.head())
display(bbox_df['label'].value_counts().rename_axis('label').reset_index(name='rows'))


## select thermal background image


In [ ]:
existing_images = []
for image_dir in THERMAL_IMAGE_DIRS:
    if image_dir.exists():
        existing_images.extend(sorted(image_dir.rglob('*.jpg')))
        existing_images.extend(sorted(image_dir.rglob('*.jpeg')))
        existing_images.extend(sorted(image_dir.rglob('*.png')))

image_by_name = {path.name: path for path in existing_images}
candidate_names = sorted(set(bbox_df['image_name'].dropna().astype(str)) & set(image_by_name))

if candidate_names:
    background_image_name = pd.Series(candidate_names).sample(1, random_state=RANDOM_STATE).iloc[0]
    BACKGROUND_IMAGE_PATH = image_by_name[background_image_name]
else:
    BACKGROUND_IMAGE_PATH = None

print(f'background image: {BACKGROUND_IMAGE_PATH}')


## load hotspot bbox annotations


In [ ]:
hotspot_rows = []
for hotspot_dir in HOTSPOT_DIRS:
    for path in sorted(hotspot_dir.glob('*.txt')):
        with path.open('r', encoding='utf-8') as handle:
            for line in handle:
                raw = line.strip()
                if not raw:
                    continue
                label, bbox, target_xy = ast.literal_eval(raw)
                label = HOTSPOT_LABEL_MAP.get(str(label), str(label))
                hotspot_rows.append({
                    'source_file': path.name,
                    'label': label,
                    'bbox_x0': float(bbox[0]),
                    'bbox_y0': float(bbox[1]),
                    'bbox_x1': float(bbox[2]),
                    'bbox_y1': float(bbox[3]),
                    'target_x': float(target_xy[0]),
                    'target_y': float(target_xy[1]),
                })

hotspot_df = pd.DataFrame(hotspot_rows)
if not hotspot_df.empty:
    hotspot_df['bbox_width'] = hotspot_df['bbox_x1'] - hotspot_df['bbox_x0']
    hotspot_df['bbox_height'] = hotspot_df['bbox_y1'] - hotspot_df['bbox_y0']
    hotspot_df['center_x'] = (hotspot_df['bbox_x0'] + hotspot_df['bbox_x1']) / 2
    hotspot_df['center_y'] = (hotspot_df['bbox_y0'] + hotspot_df['bbox_y1']) / 2

print(f'hotspot bbox rows: {len(hotspot_df):,}')
if not hotspot_df.empty:
    display(hotspot_df['label'].value_counts().rename_axis('label').reset_index(name='hotspot_rows'))
    display(hotspot_df.head())


## load manual bbox annotations


In [ ]:
manual_df = pd.read_csv(MANUAL_ANNOTATION_CSV)
manual_df = manual_df[manual_df['status'].fillna('').astype(str).str.lower().eq('ok')].copy()
manual_df = manual_df[manual_df['annotation_type'].fillna('').astype(str).str.lower().eq('bbox')].copy()
manual_df['label'] = manual_df['label'].replace(HOTSPOT_LABEL_MAP)

manual_bbox_columns = ['bbox_x0', 'bbox_y0', 'bbox_x1', 'bbox_y1']
for column in manual_bbox_columns:
    manual_df[column] = pd.to_numeric(manual_df[column], errors='coerce')
manual_df = manual_df.dropna(subset=['label', *manual_bbox_columns])
manual_df['bbox_width'] = manual_df['bbox_x1'] - manual_df['bbox_x0']
manual_df['bbox_height'] = manual_df['bbox_y1'] - manual_df['bbox_y0']
manual_df['center_x'] = (manual_df['bbox_x0'] + manual_df['bbox_x1']) / 2
manual_df['center_y'] = (manual_df['bbox_y0'] + manual_df['bbox_y1']) / 2

print(f'manual bbox rows: {len(manual_df):,}')
display(manual_df['label'].value_counts().rename_axis('label').reset_index(name='manual_rows'))
display(manual_df.head())


## bbox distribution summary


In [ ]:
summary_df = (
    bbox_df
    .groupby('label')
    .agg(
        rows=('label', 'size'),
        images=('image_name', 'nunique'),
        center_x_median=('center_x', 'median'),
        center_y_median=('center_y', 'median'),
        center_x_p05=('center_x', lambda s: s.quantile(0.05)),
        center_x_p95=('center_x', lambda s: s.quantile(0.95)),
        center_y_p05=('center_y', lambda s: s.quantile(0.05)),
        center_y_p95=('center_y', lambda s: s.quantile(0.95)),
        bbox_width_median=('bbox_width', 'median'),
        bbox_height_median=('bbox_height', 'median'),
        confidence_median=('confidence', 'median'),
    )
    .reset_index()
    .sort_values('rows', ascending=False)
)

display(summary_df)


## bbox overlay by label


In [ ]:
label_colors = {
    'Machine': 'red',
    'Light': 'orange',
    'Screen': 'blue',
    'Window': 'green',
    'Person': 'purple',
}
hotspot_color = 'cyan'
manual_color = 'yellow'

background_image = plt.imread(BACKGROUND_IMAGE_PATH) if BACKGROUND_IMAGE_PATH is not None else None
if background_image is not None:
    background_image = background_image[::-1, ::-1]


def safe_filename(value):
    return re.sub(r'[^a-z0-9]+', '_', str(value).lower()).strip('_') or 'label'


for label in sorted(bbox_df['label'].dropna().unique()):
    label_df = bbox_df[bbox_df['label'] == label].copy()
    label_hotspot_df = hotspot_df[hotspot_df['label'] == label].copy() if not hotspot_df.empty else pd.DataFrame()
    label_manual_df = manual_df[manual_df['label'] == label].copy() if not manual_df.empty else pd.DataFrame()
    plot_df = label_df
    sampled = False
    if MAX_BOXES_PER_LABEL is not None and len(label_df) > MAX_BOXES_PER_LABEL:
        plot_df = label_df.sample(MAX_BOXES_PER_LABEL, random_state=RANDOM_STATE).sort_index()
        sampled = True

    color = label_colors.get(label, 'gray')
    fig, ax = plt.subplots(figsize=(7.5, 7.5))

    if background_image is not None:
        ax.imshow(
            background_image,
            extent=[0, IMAGE_WIDTH, IMAGE_HEIGHT, 0],
            alpha=0.55,
        )

    for row in plot_df.itertuples(index=False):
        ax.add_patch(
            Rectangle(
                (row.bbox_x0, row.bbox_y0),
                row.bbox_width,
                row.bbox_height,
                fill=False,
                edgecolor=color,
                linewidth=0.55,
                alpha=0.08,
            )
        )

    for row in label_hotspot_df.itertuples(index=False):
        ax.add_patch(
            Rectangle(
                (row.bbox_x0, row.bbox_y0),
                row.bbox_width,
                row.bbox_height,
                fill=False,
                edgecolor=hotspot_color,
                linewidth=1.7,
                alpha=0.9,
                linestyle='--',
            )
        )

    for row in label_manual_df.itertuples(index=False):
        ax.add_patch(
            Rectangle(
                (row.bbox_x0, row.bbox_y0),
                row.bbox_width,
                row.bbox_height,
                fill=False,
                edgecolor=manual_color,
                linewidth=1.7,
                alpha=0.9,
                linestyle='-',
            )
        )

    ax.scatter(plot_df['center_x'], plot_df['center_y'], s=4, color=color, alpha=0.18, label='raw bbox center')
    if not label_hotspot_df.empty:
        ax.scatter(
            label_hotspot_df['center_x'],
            label_hotspot_df['center_y'],
            s=18,
            color=hotspot_color,
            alpha=0.95,
            label='hotspot bbox center',
        )
    if not label_manual_df.empty:
        ax.scatter(
            label_manual_df['center_x'],
            label_manual_df['center_y'],
            s=18,
            color=manual_color,
            alpha=0.95,
            label='manual bbox center',
        )
    ax.set_xlim(0, IMAGE_WIDTH)
    ax.set_ylim(IMAGE_HEIGHT, 0)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('bbox x coordinate')
    ax.set_ylabel('bbox y coordinate')
    shown_text = f'{len(plot_df):,}/{len(label_df):,}' if sampled else f'{len(label_df):,}'
    ax.set_title(
        f'{label} bbox overlay, raw: {shown_text}, hotspots: {len(label_hotspot_df):,}, manual: {len(label_manual_df):,}'
    )
    ax.grid(alpha=0.25)
    ax.legend(loc='best')
    fig.tight_layout()

    output_path = OUTPUT_DIR / f'v2_bbox_distribution_{safe_filename(label)}.png'
    fig.savefig(output_path, dpi=180)
    print(output_path)
    display(fig)
    plt.close(fig)


## layout constraint boundary check

plot the current allowed layout regions from `v2_config.py` on the 2d room map.

In [ ]:
# plot current layout constraint rectangles and their corner points on the 2d map.
layout_img = plt.imread(v2_config.LAYOUT_IMAGE)

region_group_colors = {
    'middle': 'tab:orange',
    'lower': 'tab:green',
    'left': 'tab:red',
    'walkway': 'tab:purple',
}

fig, ax = plt.subplots(figsize=(13, 7))
ax.imshow(
    layout_img,
    extent=[0, v2_config.MAP_WIDTH, 0, v2_config.MAP_HEIGHT],
    origin='upper',
    alpha=0.45,
)

region_rows = []
for group_name, regions in v2_config.LAYOUT_ALLOWED_REGION_GROUPS.items():
    color = region_group_colors.get(group_name, 'black')
    for region_name, x0, y0, x1, y1 in regions:
        width = x1 - x0
        height = y1 - y0
        region_rows.append({
            'group': group_name,
            'region': region_name,
            'x0': x0,
            'y0': y0,
            'x1': x1,
            'y1': y1,
            'width': width,
            'height': height,
        })
        ax.add_patch(
            Rectangle(
                (x0, y0),
                width,
                height,
                fill=False,
                edgecolor=color,
                linewidth=2.0,
                alpha=0.9,
            )
        )
        ax.scatter([x0, x1], [y0, y1], color=color, s=20, alpha=0.9)
        ax.text(x0 + 0.03, y1 + 0.03, f'{group_name}:{region_name}', color=color, fontsize=7)

for group_name, color in region_group_colors.items():
    ax.plot([], [], color=color, linewidth=2, label=group_name)

ax.set_xlim(0, v2_config.MAP_WIDTH)
ax.set_ylim(0, v2_config.MAP_HEIGHT)
ax.set_xlabel('X (meters)')
ax.set_ylabel('Y (meters)')
ax.set_title('V2 layout constraint regions')
ax.grid(alpha=0.25)
ax.legend(loc='best')
fig.tight_layout()

layout_constraint_preview = OUTPUT_DIR / 'v2_layout_constraint_regions.png'
fig.savefig(layout_constraint_preview, dpi=160, bbox_inches='tight')
print(f'wrote: {layout_constraint_preview}')
display(fig)
plt.close(fig)

region_df = pd.DataFrame(region_rows)
display(region_df.sort_values(['group', 'region']))


In [ ]:
# plot which region groups are used by each constrained label.
for label, group_names in v2_config.LAYOUT_ALLOWED_GROUPS_BY_LABEL.items():
    fig, ax = plt.subplots(figsize=(13, 7))
    ax.imshow(
        layout_img,
        extent=[0, v2_config.MAP_WIDTH, 0, v2_config.MAP_HEIGHT],
        origin='upper',
        alpha=0.45,
    )
    label_rows = []
    for group_name in group_names:
        color = region_group_colors.get(group_name, 'black')
        for region_name, x0, y0, x1, y1 in v2_config.LAYOUT_ALLOWED_REGION_GROUPS[group_name]:
            label_rows.append({
                'label': label,
                'group': group_name,
                'region': region_name,
                'x0': x0,
                'y0': y0,
                'x1': x1,
                'y1': y1,
            })
            ax.add_patch(
                Rectangle(
                    (x0, y0),
                    x1 - x0,
                    y1 - y0,
                    fill=True,
                    facecolor=color,
                    edgecolor=color,
                    linewidth=1.8,
                    alpha=0.18,
                )
            )
            ax.scatter([x0, x1], [y0, y1], color=color, s=18, alpha=0.9)
            ax.text(x0 + 0.03, y1 + 0.03, region_name, color=color, fontsize=7)

    for group_name in group_names:
        ax.plot([], [], color=region_group_colors.get(group_name, 'black'), linewidth=3, label=group_name)

    ax.set_xlim(0, v2_config.MAP_WIDTH)
    ax.set_ylim(0, v2_config.MAP_HEIGHT)
    ax.set_xlabel('X (meters)')
    ax.set_ylabel('Y (meters)')
    ax.set_title(f'V2 allowed regions for {label}')
    ax.grid(alpha=0.25)
    ax.legend(loc='best')
    fig.tight_layout()

    label_preview = OUTPUT_DIR / f'v2_allowed_regions_{safe_filename(label)}.png'
    fig.savefig(label_preview, dpi=160, bbox_inches='tight')
    print(f'wrote: {label_preview}')
    display(fig)
    plt.close(fig)
    display(pd.DataFrame(label_rows))


## hotspot target projection check

plot the hotspot target xy values used to train the v2 projection NN.

In [ ]:
# plot hotspot target xy points on the 2d room map.
hotspot_target_df = hotspot_df.copy()
for col in ['target_x', 'target_y']:
    hotspot_target_df[col] = pd.to_numeric(hotspot_target_df[col], errors='coerce')
hotspot_target_df = hotspot_target_df.dropna(subset=['label', 'target_x', 'target_y'])

fig, ax = plt.subplots(figsize=(13, 7))
ax.imshow(
    layout_img,
    extent=[0, v2_config.MAP_WIDTH, 0, v2_config.MAP_HEIGHT],
    origin='upper',
    alpha=0.45,
)

for label, group in hotspot_target_df.groupby('label'):
    ax.scatter(
        group['target_x'],
        group['target_y'],
        s=32,
        alpha=0.75,
        color=label_colors.get(label, 'gray'),
        edgecolor='black',
        linewidth=0.25,
        label=f'{label} hotspot targets ({len(group):,})',
    )

ax.set_xlim(0, v2_config.MAP_WIDTH)
ax.set_ylim(0, v2_config.MAP_HEIGHT)
ax.set_xlabel('X (meters)')
ax.set_ylabel('Y (meters)')
ax.set_title('V2 hotspot target xy used for NN projection training')
ax.grid(alpha=0.25)
ax.legend(loc='best', markerscale=1.4)
fig.tight_layout()

hotspot_target_preview = OUTPUT_DIR / 'v2_hotspot_target_xy_on_layout.png'
fig.savefig(hotspot_target_preview, dpi=160, bbox_inches='tight')
print(f'wrote: {hotspot_target_preview}')
display(fig)
plt.close(fig)

hotspot_target_summary = (
    hotspot_target_df.groupby('label')
    .agg(
        rows=('label', 'size'),
        target_x_min=('target_x', 'min'),
        target_x_median=('target_x', 'median'),
        target_x_max=('target_x', 'max'),
        target_y_min=('target_y', 'min'),
        target_y_median=('target_y', 'median'),
        target_y_max=('target_y', 'max'),
    )
    .reset_index()
    .sort_values('rows', ascending=False)
)
display(hotspot_target_summary)


In [ ]:
# plot hotspot target xy one label at a time.
for label in sorted(hotspot_target_df['label'].dropna().unique()):
    label_targets = hotspot_target_df[hotspot_target_df['label'] == label].copy()

    fig, ax = plt.subplots(figsize=(13, 7))
    ax.imshow(
        layout_img,
        extent=[0, v2_config.MAP_WIDTH, 0, v2_config.MAP_HEIGHT],
        origin='upper',
        alpha=0.45,
    )
    ax.scatter(
        label_targets['target_x'],
        label_targets['target_y'],
        s=36,
        alpha=0.75,
        color=label_colors.get(label, 'gray'),
        edgecolor='black',
        linewidth=0.25,
        label=f'{label} hotspot targets ({len(label_targets):,})',
    )
    ax.set_xlim(0, v2_config.MAP_WIDTH)
    ax.set_ylim(0, v2_config.MAP_HEIGHT)
    ax.set_xlabel('X (meters)')
    ax.set_ylabel('Y (meters)')
    ax.set_title(f'V2 hotspot target xy: {label}')
    ax.grid(alpha=0.25)
    ax.legend(loc='best')
    fig.tight_layout()

    label_preview = OUTPUT_DIR / f'v2_hotspot_target_xy_{safe_filename(label)}.png'
    fig.savefig(label_preview, dpi=160, bbox_inches='tight')
    print(f'wrote: {label_preview}')
    display(fig)
    plt.close(fig)
    display(label_targets[['source_file', 'label', 'target_x', 'target_y']].head(30))
